# Lab4.5 — Benchmark software pe ARM

În acest task consolidăm rezultatele Lab4 și construim baseline-ul software care va fi utilizat în Lab5 pentru evaluarea acceleratorului FPGA.


## Obiective

- măsurarea repetată a operațiilor grayscale, resize și Sobel;
- calculul mediei și deviației standard;
- calculul FPS teoretic;
- evaluarea dependenței de rezoluție;
- măsurarea unui pipeline software complet.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

IMAGE_DIR = Path("images")
IMAGE_DIR.mkdir(exist_ok=True)
IMAGE_PATH = IMAGE_DIR / "opencv_filters.jpg"

IMAGE_URL = (
    "https://raw.githubusercontent.com/onigas/PYNQ/master/"
    "boards/Pynq-Z1/base/notebooks/video/data/opencv_filters.jpg"
)

if not IMAGE_PATH.exists():
    print("Downloading test image ...")
    try:
        urlretrieve(IMAGE_URL, IMAGE_PATH)
    except Exception as e:
        raise RuntimeError(
            "Imaginea nu a putut fi descărcată. "
            "Copiați manual opencv_filters.jpg în Lab4/images/."
        ) from e

print("Image:", IMAGE_PATH)

In [ ]:
import cv2
import numpy as np
import time
import statistics
import matplotlib.pyplot as plt

image_bgr = cv2.imread(str(IMAGE_PATH), cv2.IMREAD_COLOR)
if image_bgr is None:
    raise RuntimeError("Imaginea nu a putut fi încărcată.")

## 1. Funcțiile testate


In [ ]:
def op_grayscale(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

def op_resize(img, size=(320, 240)):
    return cv2.resize(img, size, interpolation=cv2.INTER_LINEAR)

def op_sobel(gray):
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    return cv2.convertScaleAbs(cv2.magnitude(gx, gy))

def op_pipeline(img, size=(320, 240)):
    resized = op_resize(img, size)
    gray = op_grayscale(resized)
    return op_sobel(gray)

## 2. Funcție generică de benchmark


In [ ]:
def benchmark(func, *args, repeats=30, **kwargs):
    func(*args, **kwargs)  # warm-up
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        func(*args, **kwargs)
        times.append(time.perf_counter() - t0)
    return {
        "mean": statistics.mean(times),
        "median": statistics.median(times),
        "stdev": statistics.stdev(times) if len(times) > 1 else 0.0,
        "times": times,
    }

## 3. Benchmark la rezoluția imaginii originale

Pentru Sobel folosim o imagine grayscale pregătită înaintea măsurării, astfel încât timpul Sobel să nu includă conversia color.


In [ ]:
gray = op_grayscale(image_bgr)

tests = {
    "grayscale": benchmark(op_grayscale, image_bgr),
    "resize_320x240": benchmark(op_resize, image_bgr, (320, 240)),
    "sobel": benchmark(op_sobel, gray),
    "pipeline": benchmark(op_pipeline, image_bgr, (320, 240)),
}

print(f"{'Operation':18s} {'Mean ms':>10s} {'Std us':>10s} {'FPS':>10s}")
print("-" * 52)
for name, r in tests.items():
    print(f"{name:18s} {r['mean']*1e3:10.3f} {r['stdev']*1e6:10.3f} {1/r['mean']:10.2f}")

## 4. Benchmark pe mai multe rezoluții

Pentru fiecare rezoluție pregătim mai întâi imaginea de intrare. Pregătirea nu este inclusă în timpii operațiilor măsurate.


In [ ]:
resolutions = ((320, 240), (640, 480), (1280, 720))
results = []

for w, h in resolutions:
    img = cv2.resize(image_bgr, (w, h), interpolation=cv2.INTER_LINEAR)
    gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    r_gray = benchmark(op_grayscale, img, repeats=20)
    r_sobel = benchmark(op_sobel, gray_img, repeats=20)

    results.append((w, h, r_gray['mean'], r_sobel['mean']))
    print(
        f"{w:4d}x{h:<4d} | grayscale {r_gray['mean']*1e3:8.3f} ms | "
        f"sobel {r_sobel['mean']*1e3:8.3f} ms"
    )

## 5. Reprezentare grafică


In [ ]:
pixels = [w*h for w, h, _, _ in results]
sobel_ms = [t_sobel*1e3 for _, _, _, t_sobel in results]

plt.figure(figsize=(8, 4))
plt.plot(pixels, sobel_ms, marker="o")
plt.xlabel("Number of pixels")
plt.ylabel("Sobel time (ms)")
plt.title("Software Sobel cost vs image size")
plt.grid(True)
plt.show()

## 6. Baseline pentru Lab5

Notați rezultatele obținute pe placa utilizată:

| Operație | Rezoluție | Timp mediu [ms] | FPS |
|---|---:|---:|---:|
| Grayscale | 640×480 | | |
| Resize | 640×480 → 320×240 | | |
| Sobel | 640×480 | | |
| Pipeline Resize → Gray → Sobel | → 320×240 | | |

Aceste valori vor fi folosite ulterior pentru calculul:

```text
speed-up = T_software / T_hardware
```


## 7. Observație importantă

Timpii măsurați în acest laborator sunt timpi de **software pe ARM**, dar OpenCV este o bibliotecă nativă optimizată; nu trebuie confundată cu o implementare Python pixel-cu-pixel.

În Lab5 vom compara această implementare software realistă cu un accelerator hardware din PL.


## 8. Exercițiu

Rulați pipeline-ul complet de **50 de ori** și calculați:

- media;
- mediana;
- deviația standard;
- FPS;
- coeficientul de variație `stdev / mean`.


In [ ]:
# Scrieți aici analiza


## 9. Întrebări

1. Care operație este dominantă în pipeline-ul software măsurat?
2. Cum se modifică timpul Sobel când crește numărul de pixeli?
3. De ce trebuie să păstrăm exact aceeași operație și aceleași dimensiuni când comparăm cu FPGA-ul?
4. De ce FPS-ul unei singure operații nu este neapărat FPS-ul unei aplicații video complete?
